# Single Cell data
CC 2026-08-11

## 1. Setup

In [1]:
import retinanalysis as ra
import matplotlib.pyplot as plt
import numpy as np

# Read-only single-cell database queries and notebook browsers.
from retinanalysis.SCutils import explore as sc
# Map new h5 files when needed.
report = ra.SCutils.update_single_cell_json()

Single-cell drive: /Volumes/ChrisNewSSD
H5 folder: /Volumes/ChrisNewSSD/single_cell/chris_data/h5
JSON folder: /Volumes/ChrisNewSSD/single_cell/chris_data/json
No new H5 files; every H5 already has matching JSON.


## 2. Populate and refresh the database

`populate_database()` ingests new experiments, refreshes experiments whose JSON changed, and returns the database freshness check in the same report. Canonical experiment names such as `YYYY-MM-DD_X.h5` are included; auxiliary and legacy files are ignored. By default only metadata and tags JSON files trigger a refresh. Pass `watch_data_file=True` to include H5 modification times.

In [ ]:
# One call handles ingest, refresh, and the post-ingest stale-file check.
summary = ra.populate_database()
df_db = summary['experiments']
df_stale = summary['stale']

print(f"newly added : {len(summary['added'])}")
print(f"refreshed   : {len(summary['updated'])}")
print(f"errored     : {len(summary['skipped'])}")
print(f"database    : {len(df_db)} experiments; {len(df_stale)} still out of date")

if len(df_stale):
    display(df_stale[['exp_name', 'date_added', 'source_mtime', 'source_file']])

# ra.purge_experiments('2026-06-04_G')
# ra.purge_experiments(['2026-05-06_E', '2026-05-08_E'])

# Drop only rows that remain stale after populate.
# ra.purge_experiments(df_stale['exp_name'].tolist())

# Wipe the whole database (requires the literal confirmation token).
# ra.purge_database(confirm='YES_DELETE_ALL')

print(f'{len(df_db)} experiments currently in the database.')

## 3. List single-cell experiments

Experiments are shown in separate `chris_data` and `fred_data` tables. Each protocol gets its own row so a date is easier to scan; repeated experiment, project, and short cell-type values are visually grouped. Owner and species are kept only for the cascading browser and are omitted from the table.

In [8]:
# Section 3 has exactly these visible columns. Owner and species remain
# internal to the cascading browser and are not included here.
df_sc_exps = sc.list_experiments(show=False)

# Accept both the current singular column and an older, comma-joined
# `protocols` column from a module already cached in this kernel.
if 'protocols' in df_sc_exps.columns:
    df_sc_exps['protocol'] = df_sc_exps.pop('protocols').fillna('?').str.split(r',\s*', regex=True)
    df_sc_exps = df_sc_exps.explode('protocol', ignore_index=True)

section3_columns = ['exp_name', 'cell_types', 'protocol']
df_sc_exps = (df_sc_exps.loc[:, section3_columns]
              .drop_duplicates()
              .sort_values(['exp_name', 'protocol'], ignore_index=True))
sc.tree_table(df_sc_exps, levels=['exp_name', 'project', 'cell_types'], height=500)

exp_name,cell_types,protocol
2017-11-21_B,"OFF-transient, ON-alpha",ExpandingSpots
,,EyeMovementTrajectory
2017-12-12_B,"AII, rod bipolar",LedPulse
,,LedPulseFamily
2018-09-06_B,ON-alpha,ChirpStimulus
,,ChirpStimulusLED
,,LedPulse
2018-10-05_B,?,SingleSpot
2019-01-08_B,unknown,LedPulse
,,PulseFamily


'\n<style>\n.ra-tbl { overflow: auto; }\n.ra-tbl table { border-collapse: collapse; font-size: 12.5px;\n                font-variant-numeric: tabular-nums; }\n.ra-tbl th { position: sticky; top: 0; z-index: 1; text-align: left;\n             font-weight: 600; padding: 4px 12px 4px 0;\n             border-bottom: 1px solid rgba(128,128,128,0.6);\n             background: var(--jp-layout-color0, #fff); }\n.ra-tbl td { padding: 2px 12px 2px 0; vertical-align: top;\n             white-space: nowrap; }\n.ra-tbl tr.grp > td { border-top: 1px solid rgba(128,128,128,0.28); }\n.ra-tbl td.num { text-align: right; }\n.ra-tbl td.lead { font-weight: 600; }\n.ra-tbl summary { cursor: pointer; margin: 2px 0; }\n</style>\n<div class="ra-tbl" style="max-height:500px"><table><thead><tr><th>exp_name</th><th>cell_types</th><th>protocol</th></tr></thead><tbody><tr><td class="lead">2017-11-21_B</td><td>OFF-transient, ON-alpha</td><td>ExpandingSpots</td></tr><tr><td class="lead"></td><td></td><td>EyeMovement

### 4.1 Protocol coverage by species

One row per short protocol. Counts are unique experiment dates, not epoch blocks: `primate_dates` and `mouse_dates` show species-specific coverage, and `total_dates` includes every species.


In [2]:
# Scrollable protocol inventory, sorted by the number of dates.
df_protocol_inventory = sc.protocol_inventory(height=500)


134 protocols across 525 experiment dates.


protocol,primate_dates,mouse_dates,total_dates
SingleSpot,242,79,328
ExpandingSpots,256,57,323
SplitFieldCentering,234,37,279
VariableMeanNoise,121,16,140
LedPulse,85,38,125
ContrastReversingGrating,45,51,97
DovesMovie,57,1,59
LinearEquivalentDisc,21,37,58
JitteredNoise,46,0,51
LedNoiseFamily,40,7,48


## 4. Find experiments by protocol

Search protocol names case-insensitively. The returned DataFrame remains one row per epoch block.

In [6]:
df_blocks = sc.find_blocks('LinearEquivalentDiscConeLin')

193 blocks | 16 experiments | 1 protocol(s) matching 'LinearEquivalentDiscConeLin'


exp_name,blocks,protocols,block_ids
2026-04-10_G,3,LinearEquivalentDiscConeLin,"37324, 37332-37333"
2026-04-15_E,16,LinearEquivalentDiscConeLin,"34057-34069, 34108-34110"
2026-04-17_E,10,LinearEquivalentDiscConeLin,34210-34219
2026-04-24_E,6,LinearEquivalentDiscConeLin,34368-34373
2026-04-28_E,29,LinearEquivalentDiscConeLin,"33062-33072, 33074, 33076-33077, 33079-33081, 33084, 33087, 33092-33097, 33099, 33102, 33105-33106"
2026-05-06_E,13,LinearEquivalentDiscConeLin,"36418-36426, 36429-36432"
2026-05-06_G,3,LinearEquivalentDiscConeLin,37001-37003
2026-05-07_G,1,LinearEquivalentDiscConeLin,37121
2026-05-08_E,10,LinearEquivalentDiscConeLin,"36461, 36559-36561, 36564-36569"
2026-05-08_G,11,LinearEquivalentDiscConeLin,"37123-37125, 37129-37132, 37138-37139, 37149-37150"


exp_name,protocol,block_id
2026-04-10_G,LinearEquivalentDiscConeLin,37324
2026-04-10_G,LinearEquivalentDiscConeLin,37332
2026-04-10_G,LinearEquivalentDiscConeLin,37333
2026-04-15_E,LinearEquivalentDiscConeLin,34057
2026-04-15_E,LinearEquivalentDiscConeLin,34058
2026-04-15_E,LinearEquivalentDiscConeLin,34059
2026-04-15_E,LinearEquivalentDiscConeLin,34060
2026-04-15_E,LinearEquivalentDiscConeLin,34061
2026-04-15_E,LinearEquivalentDiscConeLin,34062
2026-04-15_E,LinearEquivalentDiscConeLin,34063


## 5. Browse and summarize experiments

Use the cascading menus to select data owner, species, and experiment. The overview is organized as cell → epoch group (group label) → protocol, with block and epoch counts. Then select an epoch block and click **Load original traces** to read and display every unprocessed Amp1 epoch trace from the H5 file.

In [10]:
experiment_browser = sc.summarize_experiments(df_sc_exps)